<a href="https://colab.research.google.com/github/fatmasenguler/Spanning-Tree_Thermostatics_of_Allostery/blob/main/6_jsd_rewiring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
6_jsd_rewiring.py
=================
Computes Jensen-Shannon divergence between WT and G12D path ensembles
and produces the residue-level rewiring (allosteric importance) table.

Reproduces: Table 3 from the manuscript.

Input: Two ensemble files generated by 5_generate_ensemble.py

Dependencies: numpy, scipy

Usage:
    python 6_jsd_rewiring.py
    # Provide WT and G12D ensemble file paths when prompted

Author: Fatma Ciftci & Burak Erman
"""

import os
import sys
from datetime import datetime
from collections import defaultdict

import numpy as np
from scipy.spatial.distance import jensenshannon


class Tee:
    def __init__(self, filename):
        self.file = open(filename, 'w', encoding='utf-8')
        self.stdout = sys.stdout

    def write(self, text):
        self.stdout.write(text)
        self.file.write(text)

    def flush(self):
        self.stdout.flush()
        self.file.flush()

    def close(self):
        self.flush()
        self.file.close()


def load_ensemble_file(filename):
    """Load a full ensemble table written by generate_path_ensemble_from_pdb.py."""
    if not os.path.isfile(filename):
        raise FileNotFoundError(f"File not found: {filename}")

    rows = []
    in_table = False
    header = None

    with open(filename, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.rstrip('\n')
            if line == "FULL ENSEMBLE TABLE":
                in_table = True
                header = None
                continue
            if not in_table:
                continue
            if not line or line.startswith("-"):
                continue
            if line.startswith("END OF ENSEMBLE") or line.startswith("="):
                break
            parts = line.split('\t')
            if header is None:
                header = parts
                continue

            if len(parts) != len(header):
                continue

            index = int(parts[0])
            probability = float(parts[-2])
            occurrences = int(parts[-1])
            residue_fields = parts[1:-2]
            path = tuple(int(x) for x in residue_fields if x.strip() != "")
            if not path:
                continue

            rows.append({
                "index": index,
                "path": path,
                "probability": probability,
                "occurrences": occurrences,
            })

    if not rows:
        raise ValueError(f"No ensemble rows loaded from {filename}")

    total_prob = float(sum(r["probability"] for r in rows))
    if total_prob <= 0:
        raise ValueError(f"Invalid total probability in {filename}")
    if abs(total_prob - 1.0) > 1e-8:
        for row in rows:
            row["probability"] /= total_prob

    return rows


def calculate_jsd_corrected(wt_rows, mt_rows):
    wt_dict = {r["path"]: r["probability"] for r in wt_rows}
    mt_dict = {r["path"]: r["probability"] for r in mt_rows}
    all_paths = sorted(set(wt_dict) | set(mt_dict))

    p = np.array([wt_dict.get(path, 0.0) for path in all_paths], dtype=np.float64)
    q = np.array([mt_dict.get(path, 0.0) for path in all_paths], dtype=np.float64)

    if p.sum() > 0:
        p = p / p.sum()
    if q.sum() > 0:
        q = q / q.sum()

    if len(all_paths) == 0:
        return 0.0
    return float(jensenshannon(p, q, base=2) ** 2)


def residue_importance(rows):
    importance = defaultdict(float)
    path_counts = defaultdict(int)
    occurrence_counts = defaultdict(int)

    for row in rows:
        path = row["path"]
        prob = row["probability"]
        unique_residues = set(path)
        for residue in path:
            occurrence_counts[residue] += 1
        for residue in unique_residues:
            importance[residue] += prob
            path_counts[residue] += 1

    return importance, path_counts, occurrence_counts


def rewiring_table(wt_rows, mt_rows):
    wt_imp, wt_path_counts, wt_occ_counts = residue_importance(wt_rows)
    mt_imp, mt_path_counts, mt_occ_counts = residue_importance(mt_rows)

    all_residues = sorted(set(wt_imp) | set(mt_imp) | set(wt_path_counts) | set(mt_path_counts))
    records = []
    for residue in all_residues:
        wt_val = wt_imp.get(residue, 0.0)
        mt_val = mt_imp.get(residue, 0.0)
        change = mt_val - wt_val
        pct_change = (change / wt_val * 100.0) if wt_val > 0 else (float('inf') if mt_val > 0 else 0.0)
        records.append({
            "residue": residue,
            "wt_importance": wt_val,
            "mt_importance": mt_val,
            "change": change,
            "pct_change": pct_change,
            "wt_paths": wt_path_counts.get(residue, 0),
            "mt_paths": mt_path_counts.get(residue, 0),
            "wt_occurrences": wt_occ_counts.get(residue, 0),
            "mt_occurrences": mt_occ_counts.get(residue, 0),
        })

    records.sort(key=lambda r: abs(r["change"]), reverse=True)
    return records


def rewiring_severity(jsd):
    if jsd < 0.05:
        return "Mild"
    if jsd < 0.10:
        return "Moderate"
    return "Severe"


def fmt_pct(value):
    if np.isinf(value):
        return "+inf%" if value > 0 else "-inf%"
    return f"{value:+8.1f}%"


def write_report(output_file, wt_file, mt_file, wt_rows, mt_rows, jsd, records, top_n=20):
    gained = sum(1 for r in records if r["change"] > 0)
    lost = sum(1 for r in records if r["change"] < 0)
    top = records[0] if records else None

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("=" * 70 + "\n")
        f.write("ENSEMBLE REWIRING ANALYSIS\n")
        f.write("=" * 70 + "\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"WT ensemble file: {wt_file}\n")
        f.write(f"MT ensemble file: {mt_file}\n")
        f.write(f"WT total paths: {len(wt_rows)}\n")
        f.write(f"MT total paths: {len(mt_rows)}\n")
        f.write(f"WT total probability: {sum(r['probability'] for r in wt_rows):.8f}\n")
        f.write(f"MT total probability: {sum(r['probability'] for r in mt_rows):.8f}\n")

        f.write("\n" + "=" * 70 + "\n")
        f.write("RESIDUE REWIRING HOTSPOTS\n")
        f.write("=" * 70 + "\n\n")

        header = f"{'Rank':<6} {'Residue':<10} {'WT Importance':<16} {'MT Importance':<16} {'Change':<12} {'% Change':<10}"
        f.write(header + "\n")
        f.write("-" * 80 + "\n")
        for rank, r in enumerate(records[:top_n], start=1):
            sign = "+" if r['change'] >= 0 else ""
            f.write(
                f"{rank:<6} {r['residue']:<10} {r['wt_importance']:<16.6f} {r['mt_importance']:<16.6f} "
                f"{sign}{r['change']:<11.6f} {fmt_pct(r['pct_change']):<10}\n"
            )

        f.write("\n" + "=" * 70 + "\n")
        f.write("INTERPRETATION SUMMARY\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"• JSD = {jsd:.4f}: {rewiring_severity(jsd)} rewiring\n")
        f.write(f"• {gained} residues gained importance, {lost} lost importance\n")
        if top is not None:
            if top['change'] >= 0:
                f.write(f"• Residue {top['residue']} is the primary rewiring hub (+{top['change']:.4f})\n")
            else:
                f.write(f"• Residue {top['residue']} lost the most importance ({top['change']:.4f})\n")

        f.write("\n" + "=" * 70 + "\n")
        f.write("ADDITIONAL RESIDUE DETAILS\n")
        f.write("=" * 70 + "\n\n")
        f.write(
            f"{'Rank':<6} {'Residue':<10} {'WT_Imp':<12} {'MT_Imp':<12} {'WT_Paths':<10} {'MT_Paths':<10} "
            f"{'WT_Occ':<10} {'MT_Occ':<10}\n"
        )
        f.write("-" * 90 + "\n")
        for rank, r in enumerate(records, start=1):
            f.write(
                f"{rank:<6} {r['residue']:<10} {r['wt_importance']:<12.6f} {r['mt_importance']:<12.6f} "
                f"{r['wt_paths']:<10} {r['mt_paths']:<10} {r['wt_occurrences']:<10} {r['mt_occurrences']:<10}\n"
            )


def main():
    print("=" * 80)
    print("JENSEN-SHANNON REWIRING ANALYSIS FROM ENSEMBLE FILES")
    print("=" * 80)

    wt_file = input("WT ensemble file: ").strip()
    mt_file = input("MT ensemble file: ").strip()
    output_default = "rewiring_analysis_from_ensembles.txt"
    output_file = input(f"Output file [{output_default}]: ").strip() or output_default

    print("\nLoading ensembles...")
    wt_rows = load_ensemble_file(wt_file)
    mt_rows = load_ensemble_file(mt_file)
    print(f"  WT rows: {len(wt_rows)}")
    print(f"  MT rows: {len(mt_rows)}")

    print("\nComputing Jensen-Shannon divergence and rewiring table...")
    jsd = calculate_jsd_corrected(wt_rows, mt_rows)
    records = rewiring_table(wt_rows, mt_rows)
    print(f"  JSD: {jsd:.8f}")
    if records:
        print(f"  Strongest change: residue {records[0]['residue']} ({records[0]['change']:+.6f})")

    print("\nWriting report...")
    write_report(output_file, wt_file, mt_file, wt_rows, mt_rows, jsd, records)
    print(f"Done. Report written to: {os.path.abspath(output_file)}")


if __name__ == "__main__":
    tee = None
    try:
        log_filename = f"rewiring_from_ensembles_console_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        tee = Tee(log_filename)
        sys.stdout = tee
        main()
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if tee is not None:
            sys.stdout = tee.stdout
            tee.close()


JENSEN-SHANNON REWIRING ANALYSIS FROM ENSEMBLE FILES
WT ensemble file: 5_ensemble_6GOD_55_to_60_nodes_3_to_9.txt
MT ensemble file: 5_ensemble_6GOF_55_to_60_nodes_3_to_9.txt
Output file [rewiring_analysis_from_ensembles.txt]: rewiring_from_ensembles_console_20260428_055520.txt

Loading ensembles...
  WT rows: 638606
  MT rows: 642205

Computing Jensen-Shannon divergence and rewiring table...
  JSD: 0.00025030
  Strongest change: residue 59 (-0.007200)

Writing report...
Done. Report written to: /content/rewiring_from_ensembles_console_20260428_055520.txt
